# Planetary Spectra: Fetch, Cache, and Temporal Compare

This notebook demonstrates:
- fetching two timed spectra for the same target
- caching and database persistence
- temporal comparison metrics and visualization

In [9]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

import matplotlib.pyplot as plt

from stellar_spectrospy.planetary_spectra.planetary_runner import PlanetaryRunner

In [10]:
runner = PlanetaryRunner(
    cache_dir=Path("stellar_spectrospy/planetary_spectra/spectral_cache"),
    db_path=Path("stellar_spectrospy/planetary_spectra/planetary_results.db"),
)

target = "Mars"
date_a = "2026-03-01"
date_b = "2026-03-10"

In [11]:
result = runner.compare_two_dates(
    target_name=target,
    date_a=date_a,
    date_b=date_b,
    source_priority=["cache", "local_csv", "nasa_pds", "hitran", "synthetic"],
)

result["temporal"]

[Spectrum loaded]  points=6001  λ=[3500.0, 9500.0] Å  name='Mars'
[Preprocess]  method=Savitzky-Golay
[Transform]  mode=stellar  freq_bins=6001
[Peaks]  detected=2
[Spectrum loaded]  points=6001  λ=[3500.0, 9500.0] Å  name='Mars'
[Preprocess]  method=Savitzky-Golay
[Transform]  mode=stellar  freq_bins=6001
[Peaks]  detected=2


{'spectral_difference_rms': 0.0,
 'detected_shift_angstrom': 0.0,
 'harmonic_delta_l2': 0.0,
 'median_flux_delta': 0.0}

In [12]:
spec_a = runner.db.get_spectrum(result["spectrum_id_a"])
spec_b = runner.db.get_spectrum(result["spectrum_id_b"])

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Overlay spectra
ax[0].plot(spec_a["wavelength"], spec_a["flux"], linewidth=2, label=f"{target} {date_a}", color='steelblue')
ax[0].plot(spec_b["wavelength"], spec_b["flux"], linewidth=2, label=f"{target} {date_b}", alpha=0.75, color='coral')
ax[0].set_title("Timed Spectra Comparison", fontsize=12, fontweight='bold')
ax[0].set_xlabel("Wavelength (Angstrom)", fontsize=11)
ax[0].set_ylabel("Normalized Flux", fontsize=11)
ax[0].legend(fontsize=10)
ax[0].grid(True, alpha=0.3, linestyle='--')

# Right plot: Flux difference
common = spec_a["wavelength"]
delta = spec_b["flux"] - spec_a["flux"]
ax[1].fill_between(common, delta, alpha=0.4, color='tab:red')
ax[1].plot(common, delta, color="darkred", linewidth=2)
ax[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
ax[1].set_title("Flux Difference (date_b - date_a)", fontsize=12, fontweight='bold')
ax[1].set_xlabel("Wavelength (Angstrom)", fontsize=11)
ax[1].set_ylabel("Delta Flux", fontsize=11)
ax[1].grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print(f"Spectra range: {common[0]:.1f} - {common[-1]:.1f} Å")
print(f"Max flux change: {delta.max():.4f}, Min flux change: {delta.min():.4f}")

Spectra range: 3500.0 - 9500.0 Å
Max flux change: 0.0000, Min flux change: 0.0000


C:\Users\Windows User\AppData\Local\Temp\ipykernel_23800\3725162602.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
history = runner.get_target_history(target)
history[:3]

[{'id': 1,
  'object_name': 'Mars',
  'object_type': 'planet',
  'observation_date': '2026-03-01',
  'query_timestamp': '2026-03-15T05:05:44.256556',
  'source': 'hitran',
  'metadata_json': {'object_type': 'planet',
   'query_timestamp': '2026-03-15T05:05:44.226842'},
  'timestamp': '2026-03-15T05:05:44.266844+00:00'},
 {'id': 3,
  'object_name': 'Mars',
  'object_type': 'planet',
  'observation_date': '2026-03-01',
  'query_timestamp': '2026-03-15T05:52:25.250600',
  'source': 'csv_cache',
  'metadata_json': {'object_type': 'planet',
   'query_timestamp': '2026-03-15T05:52:25.192350'},
  'timestamp': '2026-03-15T05:52:25.332402+00:00'},
 {'id': 2,
  'object_name': 'Mars',
  'object_type': 'planet',
  'observation_date': '2026-03-10',
  'query_timestamp': '2026-03-15T05:05:44.298593',
  'source': 'hitran',
  'metadata_json': {'object_type': 'planet',
   'query_timestamp': '2026-03-15T05:05:44.271357'},
  'timestamp': '2026-03-15T05:05:44.302720+00:00'}]

In [14]:
export_path = runner.export_temporal_analysis(
    target_name=target,
    output_csv=Path("stellar_spectrospy/planetary_spectra/planetary_temporal_results.csv"),
)
export_path

WindowsPath('stellar_spectrospy/planetary_spectra/planetary_temporal_results.csv')